# GANrec on Real APS Beamtime Data (cfg59)

Companion to [ganrec_phantom_walkthrough.ipynb](ganrec_phantom_walkthrough.ipynb). Where the phantom notebook explains *how* GANrec works against a synthetic shepp3D, this one applies it to real ptycho-tomography projections from the Oct 2025 APS beamtime and compares against the project's standard gridrec baseline.

**Two entry points:**
- `USE_PRE_ALIGNED = True` — load the already-aligned 4×-downsampled TIFF at [alignedProjections/APSbeamtime_Oct25/cfg59_ds4x_aligned_20260505-121404.tif](alignedProjections/APSbeamtime_Oct25/cfg59_ds4x_aligned_20260505-121404.tif). Fast — skip straight to reconstruction.
- `USE_PRE_ALIGNED = False` — load the raw HDF5, drop the two known-bad projections, 4× downsample, then run the same XCA + PMA alignment pipeline as [test_notebook_realData.ipynb](test_notebook_realData.ipynb).

**Sections:**
1. Setup and config flags
2. Load (or align) projections + angles
3. Baseline reconstruction — `gridrec` on the full volume
4. Brief ganrec recap — Generator + Radon on a fresh slice (the deep walkthrough is in the phantom notebook)
5. Full `GANtomo(...).recon()` call on one slice
6. Side-by-side comparison (gridrec vs ganrec) + sinogram-consistency check
7. (Optional) Full 3-D ganrec loop — slow

## 1. Setup

In [ ]:
import os
import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import zoom

from tomoDataClass import tomoData
from helperFunctions import MoviePlotter, convert_to_numpy

from ganrectorch.ganrec import GANtomo
from ganrectorch.propagators import RadonTransform
from ganrectorch.models import Generator

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
USE_PRE_ALIGNED = True              # False → run alignment pipeline from scratch

ALIGNED_TIFF = '/home/ljh79/TomoMono/alignedProjections/APSbeamtime_Oct25/cfg59_ds4x_aligned_20260505-121404.tif'
RAW_HDF5     = '/home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2.hdf5'
DS4_CACHE    = '/home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2_ds4.hdf5'

DOWNSAMPLE = 4                      # only used when aligning from scratch
BAD_PROJS  = [26, 19]               # indices in the original (558,…) array to remove (matches realData notebook)

ITER_NUM   = 1000                   # GAN iterations per slice
RECON_ALG  = 'gridrec'              # baseline algorithm for the volume reconstruction
SLICE_IDX  = None                   # None → use centre slice; or set an integer
RNG_SEED   = 0

torch.manual_seed(RNG_SEED); np.random.seed(RNG_SEED)

## 2. Load (or align) the projections + angles

The two branches below produce the same final state: a `tomoData` instance whose `finalProjections` (and `workingProjections`) are the aligned projections, and `tomo.ang` are the centred angles in radians.

In [ ]:
def _load_angles_from_hdf5(path, drop_indices=None):
    """Read just the angles from the source HDF5 and centre them around 0 (radians)."""
    with h5py.File(path, 'r') as hf:
        ang_deg = hf['angles'][...]
    if drop_indices:
        ang_deg = np.delete(ang_deg, drop_indices)
    ang_rad = ang_deg * np.pi / 180.0
    return ang_rad - np.mean(ang_rad)

if USE_PRE_ALIGNED:
    print(f'Loading pre-aligned TIFF: {ALIGNED_TIFF}')
    projections, _ = convert_to_numpy(ALIGNED_TIFF)
    projections = projections.astype(np.float32)
    print(f'  projections shape (n_angles, h, w): {projections.shape}')

    # The aligned TIFF holds all 558 original projections (no bad-projection removal applied).
    # If yours was saved differently, adjust drop_indices below.
    angles = _load_angles_from_hdf5(RAW_HDF5, drop_indices=None)
    assert len(angles) == projections.shape[0], (
        f'angle count {len(angles)} ≠ projection count {projections.shape[0]}. '
        f'Adjust drop_indices in _load_angles_from_hdf5 to match the TIFF.'
    )
    print(f'  angles loaded from HDF5: {len(angles)}  '
          f'range [{np.degrees(angles.min()):.2f}, {np.degrees(angles.max()):.2f}] deg')

    tomo = tomoData(projections, angles)
    tomo.finalProjections = projections.copy()
    tomo.workingProjections = projections.copy()

else:
    # ── Align-from-scratch branch (mirrors test_notebook_realData.ipynb) ─
    print('Loading raw HDF5 and running alignment pipeline …')
    src = DS4_CACHE if os.path.exists(DS4_CACHE) else RAW_HDF5
    print(f'  source: {src}')

    with h5py.File(src, 'r') as hf:
        projs_og = hf['data'][...]
        ang_deg  = hf['angles'][...]
    print(f'  raw shape: {projs_og.shape}')

    if src == RAW_HDF5:
        # Drop bad projections (test_notebook_realData.ipynb removes indices 26 then 19 in that order)
        for idx in BAD_PROJS:
            projs_og = np.delete(projs_og, idx, axis=0)
            ang_deg  = np.delete(ang_deg,  idx)
        print(f'  after dropping {BAD_PROJS}: {projs_og.shape}')

        projections = zoom(projs_og, (1, 1/DOWNSAMPLE, 1/DOWNSAMPLE), order=1).astype(np.float32)
        del projs_og
    else:
        projections = projs_og.astype(np.float32)

    print(f'  downsampled shape: {projections.shape}')
    angles = ang_deg * np.pi / 180.0
    angles = angles - np.mean(angles)

    tomo = tomoData(projections, angles)
    tomo.reset_workingProjections(x_size=None, y_size=None)
    tomo.normalize(isPhaseData=True)

    # Pass 1 — full frame, no ROI
    tomo.cross_correlate_align(tolerance=0, maxShiftTolerance=0,
                               yROI_Range=None, xROI_Range=None, isFull360=False,
                               downsample=4, max_iterations=10, stepRatio=0.8, use_grad=True, plot=False)

    # ROI for passes 2+ (compute_roi formula from hyperparameter_search.py)
    H = tomo.workingProjections.shape[1]
    W = tomo.workingProjections.shape[2]
    edge  = 200 // DOWNSAMPLE
    inner = W - 2 * edge
    xca_yROI = np.array([int(0.05 * H), int(0.85 * H)])
    xca_xROI = np.array([edge + int(0.15 * inner), edge + int(0.85 * inner)])
    print(f'  XCA ROI — y: {xca_yROI.tolist()}  x: {xca_xROI.tolist()}')

    tomo.cross_correlate_align(tolerance=0, maxShiftTolerance=0,
                               yROI_Range=xca_yROI, xROI_Range=xca_xROI, isFull360=False,
                               downsample=2, max_iterations=10, stepRatio=0.8, use_grad=True, plot=False)
    tomo.cross_correlate_align(tolerance=0, maxShiftTolerance=0,
                               yROI_Range=xca_yROI, xROI_Range=xca_xROI, isFull360=False,
                               downsample=1, max_iterations=5,  stepRatio=0.8, use_grad=True, plot=False)

    # PMA — 3-level pyramid, scale=2, [2,2,2] iters, sigma=2.0
    tomo.PMA(tolerance=0, algorithm=RECON_ALG, plot=False,
             levels=3, scale=2, iterations_per_level=[2, 2, 2],
             shift_method='optical_flow', of_sigma=2.0, stepRatio=0.8)

    tomo.make_updates_shift()
    print('Alignment pipeline complete.')
    projections = tomo.finalProjections

print(f'\nReady — projections {projections.shape}, angles {tomo.ang.shape}')
print(f'Angle span: {np.degrees(tomo.ang.max() - tomo.ang.min()):.2f}° covered  '
      f'→ ~{180 - np.degrees(tomo.ang.max() - tomo.ang.min()):.0f}° missing wedge')

In [ ]:
if SLICE_IDX is None:
    SLICE_IDX = projections.shape[1] // 2

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(projections[projections.shape[0] // 2], cmap='gray', aspect='auto')
ax[0].set_title(f'Centre projection (frame {projections.shape[0] // 2})')
ax[1].imshow(projections[:, SLICE_IDX, :], cmap='gray', aspect='auto')
ax[1].set_title(f'Sinogram for slice z={SLICE_IDX}'); ax[1].set_xlabel('detector px'); ax[1].set_ylabel('projection #')
plt.tight_layout(); plt.show()

## 3. Baseline reconstruction — tomopy `gridrec` on the full volume

In [ ]:
tomo.reconstruct(algorithm=RECON_ALG)
gridrec_vol = tomo.recon
print(f'{RECON_ALG} volume shape: {gridrec_vol.shape}')

cy, cz = gridrec_vol.shape[1] // 2, gridrec_vol.shape[0] // 2
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(gridrec_vol[SLICE_IDX], cmap='gray'); ax[0].set_title(f'XY  z={SLICE_IDX}')
ax[1].imshow(gridrec_vol[:, cy, :], cmap='gray', aspect='auto'); ax[1].set_title(f'XZ  y={cy} — note missing-wedge elongation')
ax[2].imshow(gridrec_vol[:, :, gridrec_vol.shape[2] // 2], cmap='gray', aspect='auto'); ax[2].set_title('YZ — vertical-axis view')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 4. Brief GANrec recap — Generator + Radon on this slice

Just enough to confirm the components run on real data and to set expectations for what the GAN training loop is minimising. For a full Generator / Discriminator / loss / unrolled-loop walkthrough, see [ganrec_phantom_walkthrough.ipynb](ganrec_phantom_walkthrough.ipynb).

In [ ]:
def torch_nor_tomo(x):
    x = (x - x.mean()) / x.std()
    return x - x.min()

prj_slice  = projections[:, SLICE_IDX, :].astype(np.float32)              # (n_angles, width)
angles_rad = tomo.ang.astype(np.float32)                                  # radians, centred
n_ang, px  = prj_slice.shape
print(f'Slice sinogram shape: {prj_slice.shape}  (n_angles ≥ ~110 required by Discriminator)')

prj_t = torch_nor_tomo(torch.from_numpy(prj_slice).view(1, 1, n_ang, px).to(DEVICE))
ang_t = torch.from_numpy(angles_rad).to(DEVICE)

torch.manual_seed(RNG_SEED)
gen   = Generator(img_h=n_ang, img_w=px, conv_num=32, conv_size=3, dropout=0.25, output_num=1).to(DEVICE)
radon = RadonTransform(torch.empty(1, 1, px, px, device=DEVICE), ang_t).to(DEVICE)

with torch.no_grad():
    init_recon = torch_nor_tomo(gen(prj_t))
    rep        = torch_nor_tomo(radon(init_recon, ang_t))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(prj_t[0, 0].cpu(), cmap='gray', aspect='auto'); ax[0].set_title('Measured sinogram')
ax[1].imshow(rep[0, 0].cpu(),   cmap='gray', aspect='auto'); ax[1].set_title('Reprojection of untrained Generator')
ax[2].imshow(init_recon[0, 0].cpu(), cmap='gray'); ax[2].set_title('Generator output (random init)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
print('The L1 part of GANtomo\'s loss minimises |measured − reprojection| over the training iterations.')

## 5. Run `GANtomo` on the slice

`recon_monitor=False` keeps the cell quiet — no live matplotlib display from `RECONmonitor`.

In [ ]:
torch.manual_seed(RNG_SEED)
ganrec_slice = GANtomo(prj_slice, angles_rad, iter_num=ITER_NUM, recon_monitor=False).recon()
ganrec_slice = np.asarray(ganrec_slice)
while ganrec_slice.ndim > 2:
    ganrec_slice = ganrec_slice[0]
print(f'ganrec slice shape: {ganrec_slice.shape}')

## 6. Compare — gridrec vs ganrec

We have no ground truth on real data, so we compare two ways:

1. **Visual** — XY slice side by side, plus the XZ profile from the gridrec volume showing the missing-wedge smear.
2. **Sinogram consistency** — forward-project both reconstructions of the slice through the same `RadonTransform` and measure the L1 distance to the measured sinogram. A lower number means the reconstruction better explains the data.

In [ ]:
def zscore(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

gridrec_s = gridrec_vol[SLICE_IDX]
gr_n, gn_n = zscore(gridrec_s), zscore(ganrec_slice)

vmin, vmax = np.percentile(gr_n, [2, 98])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(gr_n, cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title(f'{RECON_ALG} — z={SLICE_IDX}')
ax[1].imshow(gn_n, cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title(f'ganrec — z={SLICE_IDX}  ({ITER_NUM} iters)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

# Reprojection-consistency check on this slice
def reproject(img_2d, angles_rad_t):
    t = torch.from_numpy(img_2d.astype(np.float32)).view(1, 1, *img_2d.shape).to(DEVICE)
    radon_local = RadonTransform(torch.empty(1, 1, img_2d.shape[0], img_2d.shape[1], device=DEVICE),
                                 angles_rad_t).to(DEVICE)
    with torch.no_grad():
        r = torch_nor_tomo(radon_local(torch_nor_tomo(t), angles_rad_t))
    return r[0, 0].cpu().numpy()

meas_n  = (prj_slice - prj_slice.mean()) / prj_slice.std()
meas_n -= meas_n.min()

rep_gr = reproject(gridrec_s,    ang_t)
rep_gn = reproject(ganrec_slice, ang_t)

l1_gr = float(np.mean(np.abs(meas_n - rep_gr)))
l1_gn = float(np.mean(np.abs(meas_n - rep_gn)))
print(f"{'Method':<10}  {'L1(reprojection − measured) ↓':>32}")
print(f"{'-' * 46}")
print(f"{RECON_ALG:<10}  {l1_gr:>32.4f}")
print(f"{'ganrec':<10}  {l1_gn:>32.4f}")

**Interpreting the comparison.** On real data with a missing wedge, `gridrec` tends to be sharp inside the sampled angular range but smears structures along the unsampled direction (visible most clearly in the XZ panel of Section 3). `ganrec`'s implicit prior tends to suppress that smear; in exchange, fine high-frequency detail can be slightly softer until the GAN training has run long enough (try bumping `ITER_NUM` to 2000 if convergence looks incomplete). The reprojection-L1 numbers above give a *data-fidelity* score — they measure how well each reconstruction explains the measured sinogram. Lower is better, but a too-low number on a noisy dataset is a sign of overfit, not better truth.

## 7. (Optional) Full 3-D ganrec reconstruction

GANtomo is 2-D, so a full volume means looping over every Y slice. For the 4× downsampled data (`projections.shape[1]` slices, ~146 for cfg59) at `ITER_NUM=1000` on one GPU this is typically **~1–4 hours**. Skip on the first run.

In [ ]:
# ⚠ SLOW — uncomment to run the full 3-D ganrec reconstruction.
#
# from tqdm.notebook import tqdm as tqdm_nb
#
# n_slices, _, w = projections.shape[1], projections.shape[0], projections.shape[2]
# ganrec_vol = np.zeros((projections.shape[1], w, w), dtype=np.float32)
# for y in tqdm_nb(range(projections.shape[1]), desc='ganrec slices'):
#     sino = projections[:, y, :].astype(np.float32)
#     out = GANtomo(sino, angles_rad, iter_num=ITER_NUM, recon_monitor=False).recon()
#     out = np.asarray(out)
#     while out.ndim > 2:
#         out = out[0]
#     ganrec_vol[y] = out
#
# print('ganrec volume shape:', ganrec_vol.shape)
# MoviePlotter(ganrec_vol)